In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# TOPIC 1

## 1) (Score: 15%) Briefly write the steps (with relevant formulas) that you would perform to estimate the means μ(1) 1 and μ(1) 2 , variances σ21(1) and σ22(1), and the mixing coefficients π(1) 1 and π(1) 2 of the GMM model during the first iteration of the EM algorithm.

The first iteration of the EM algorithm would go as follows:
1. Initialize the means $\mu_k$, covariances $\Sigma_k$ and mixing coefficients $\pi_k$, and evaluate the initial value of the log likelihood.
2. E step. Evaluate the responsibilities using the current parameter values
3. M step. Re-estimate the parameters using the current responsibilities
4. Evaluate the log likelihood and check for convergence of either the parameters or the log likelihood. If the convergence criterion is not satisfied return to step 2.

## 2) (Score: 60%) Using the above steps, perform a single iteration of the EM algorithm on the given dataset, using the initial parameters above, to compute the updated model parameters (means μ(1) 1 and μ(1) 2 , variances (σ21(1) and σ22(1), and mixing coefficients (π(1) 1 and π(1) 2 ). Show the relevant steps (or code) and the updated parameter values.

In [2]:
def gaussian_mixture(x, means, covariances, pis):
    K = len(means)
    result = 0

    for k in range(K):
        result += pis[k] * multivariate_normal.pdf(x, mean = means[k], cov = covariances[k])
        
    return result

def E_step(mus_0, covariances_0, pis_0, data_x, K, N):
    gamma = np.zeros((N, K))
    
    for n in range(N):
        for k in range(K):
            numerator = pis_0[k] * multivariate_normal.pdf(data_x[n], mean = mus_0[k], cov = covariances_0[k])
            denominator = gaussian_mixture(data_x[n], mus_0, covariances_0, pis_0)
            gamma[n,k] = numerator / denominator

    return gamma

def M_step(mus_0, covariances_0, pis_0, data_x, gamma, K, N):
    mu_new = np.zeros_like(mus_0)
    covariance_new = np.zeros_like(covariances_0)
    pi_new = np.zeros_like(pis_0)

    for k in range(K):
        N_k = np.sum(gamma[:, k])
        mu_new[k] = 1/N_k * np.sum(gamma[:, k][:, np.newaxis] * data_x, axis = 0)
        for n in range(N):
            covariance_new[k] += gamma[n, k] * ((data_x[n] - mu_new[k]) @ (data_x[n] - mu_new[k]).T)
        covariance_new[k] /= N_k
        pi_new[k] = N_k / N

    return mu_new, covariance_new, pi_new

def log_likelihood(mu, covariance, pi, data_x, N):
    loglikelihood = 0

    for n in range(N):
        loglikelihood += np.log(gaussian_mixture(data_x[n], mu, covariance, pi))

    return loglikelihood

def expectation_maximization_gaussian(mus_0, covariances_0, pis_0, data_x,
                                        max_iters=200, tol=1e-6, reg=1e-6):
    K = len(mus_0)
    N, _ = data_x.shape
    steps = []
    loglikelihood_prev = 0

    for i in range(max_iters):
        # Calculate gamma (step 2)
        gamma = E_step(mus_0, covariances_0, pis_0, data_x, K, N)

        # Update parameters (step 3)
        mu_new, covariance_new, pi_new = M_step(mus_0, covariances_0, pis_0, data_x, gamma, K, N)

        # Calculate log-likelihood (step 4)
        loglikelihood = log_likelihood(mu_new, covariance_new, pi_new, data_x, N)

        # Make prediction and store the results
        prediction = np.argmax(gamma, axis=1)
        steps.append((mu_new, covariance_new, pi_new, loglikelihood, prediction))

        # Check for convergence
        if np.abs(loglikelihood - loglikelihood_prev) < tol:
            break
        elif np.allclose(mus_0, mu_new, atol=tol) and np.allclose(covariances_0, covariance_new, atol=tol) and np.allclose(pis_0, pi_new, atol=tol):
            break

        # Update old parameters and get ready for next iteration
        mus_0, covariances_0, pis_0, loglikelihood_prev = np.copy(mu_new), np.copy(covariance_new), np.copy(pi_new), loglikelihood

    return steps

In [3]:
x = np.array([[1.0, 1.5, 1.2, 5.8, 6.2, 6.5]]).T
mus_0 = np.array([2.0, 3.0])
covariances_0 = np.array([0.1, 0.1])
pis_0 = np.array([0.5, 0.5])

mus, covariances, pis, loglikelihood, prediction = expectation_maximization_gaussian(mus_0, covariances_0, pis_0, x, max_iters=1)[0]
print("mus:", mus)
print("vars:", covariances)
print("pis:", pis)

mus: [1.23332935 6.16659178]
vars: [0.04222181 0.08257176]
pis: [0.49999201 0.50000799]


C:\Users\henri\AppData\Local\Temp\ipykernel_9632\321762774.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mu_new[k] = 1/N_k * np.sum(gamma[:, k][:, np.newaxis] * data_x, axis = 0)


## 3) (Score: 15%) Given the input data points x and your obtained estimates in the previous step, comment on whether the updated mean estimates appear to be converging toward values that adequately represent different clusters in the dataset.

It can be seen that the data is split into two clusters. One around 1.25 and one around 6.2. It seems that the mean values are converging toward this value after only one iteration reaching 1.233 and 6.167. The variance also went down as they are closer to the mean of the clusters which are not that spread.

## 4) (Score: 10%) Explain why an iterative approach (EM algorithm) is required to estimate the GMM model parameters using the Maximum Likelihood (ML) approach, unlike a standard Gaussian case where model parameters are obtained directly.


In a standard gaussian you only have one mean and variance, which describes the entire dataset. When there is a mixture of gaussians you have multiple, and you don't necessarily know which part of the data belongs to which clustser. You therefore have to iterate through algorithm to get closer and closer to finding the clusters instead of just knowing where there are to start with.

# TOPIC 2

## 1) (Score: 50 %) Assume that the class conditional probability densities, i.e., p(x|C1) and p(x|C2) are Gaussian with means μ1 and μ2 respectively and the same covariance matrix Σ. Further, let p(C1) and p(C2) denote the class priors for the classes C1 and C2, respectively. Using the training data points (blue circles and red crosses in Figure 1 and the maximum likelihood approach, calculate the values of the parameters μ1, μ2, Σ, p(C1), and p(C2) for the probabilistic generative model.

In [4]:
def PGM_MLE_mean(x, l, ns, C):
    K = len(C)
    mus = []
    for k in range(K):
        mu = 1/ns[k] * np.sum(x[:,l==C[k]], axis=1)[:, np.newaxis]
        mus.append(mu)
    return mus


def PGM_MLE_covariance(x, l, ns, C, mus, N):
    K = len(C)
    Ss = []
    for k in range(K):
        S = 1/ns * ((x[:,l==C[k]] - mus[k]) @ (x[:,l==C[k]] - mus[k]).T)
        Ss.append(S)
    
    sigma = 0
    for n, S in zip(ns, Ss):
        sigma += n/N * S

    return sigma

def PGM_MLE_probability(N, ns):
    Ps = []
    for n in ns:
        Ps.append(n/N)
    return Ps

def PGM_MLE_parameters(data, labels):
    classes, ns = np.unique(labels, return_counts=True)
    N = len(data.T)

    mus = PGM_MLE_mean(data, labels, ns, classes)
    sigma = PGM_MLE_covariance(data, labels, ns, classes, mus, N)
    probs = PGM_MLE_probability(N, ns)

    return mus, sigma, probs

In [5]:
X = np.array([  [2, 4, 4, 6, 7, 7],
                [3, 5, 3, 6, 7, 5]])
T = np.array([  [5],
                [4.5]])
l = np.array([0, 0, 0, 1, 1, 1])
label_names = {
    0: 2,
    1: 1
}

mus, sigma, probs = PGM_MLE_parameters(X, l)

for i, mu in enumerate(mus):
    print(f"mu_{i} =\n{mu}\n")

print(f"sigma =\n{sigma}\n")

for i, p in enumerate(probs):
    print(f"p(c_{label_names[i]}) = {p}\n")

mu_0 =
[[3.33333333]
 [3.66666667]]

mu_1 =
[[6.66666667]
 [6.        ]]

sigma =
[[0.55555556 0.22222222]
 [0.22222222 0.77777778]]

p(c_2) = 0.5

p(c_1) = 0.5



## 2) (Score: 25 %) Using the estimated parameter values in the previous step, compute the posterior probability function p(C1|x).

In [ ]:
def sigmoid(a):
    return 1/(1 + np.exp(-a))

# FOR ACTUAL GENERAL FORMULA USE: (4.62) page 216 and (4.68), (4.69), (4.70) page 217
def w(mus, cov):
    ws = []
    for mu in mus:
        ws.append(np.linalg.inv(cov) @ mu)
    return ws

def w_0(mus, cov, ps):
    w_0s = []
    for mu, p in zip(mus, ps):
        w_0s.append(- 1/2 * mu.T @ np.linalg.inv(cov) @ mu + np.log(p))
    return w_0s

def A(ws, w_0s, x):
    As = []
    for w, w_0 in zip(ws, w_0s):
        As.append(w.T @ x + w_0)
    return As

def probability_given_x(As):
    probs = []
    for a in As:
        probs.append(np.exp(a) / np.sum(np.exp(As)))
    return probs

ws = w(mus, sigma)
w_0s = w_0(mus, sigma, probs)
As = A(ws, w_0s, T)
probs_x = probability_given_x(As)
p0 = probs_x[0].item()

0.6186615298491829

## 3) (Score: 25 %) Estimate and plot the decision boundary of the obtained probabilistic generative classifier, together with the available data points in Figure 1. Which class will you assign to the test data point (green triangle) using the obtained decision boundary?


# Topic 3

## 1) (Score: 10%) Can you apply Fisher’s linear discriminant model directly to input data (i.e., without using nonlinear basis functions ϕ(x)) to correctly classify the test data points into the two classes, C1 and C2? If yes, briefly explain how that can be done. If not, why not?

## 2) (Score: 10%) Can you apply Logistic regression model directly to input data (i.e., without using nonlinear basis functions ϕ(x)) to correctly classify the test data points into the two classes, C1 and C2? If yes, briefly explain how that can be done. If not, why not?

## 3) (Score: 10%) Can you apply Linear Support Vector Machines that uses linear kernel κ(x, x′) = xT x′ to classify the test data points into the two classes, C1 and C2, with high classification accuracy. If yes, briefly explain how that can be done. If not, why not?

## 4) (Score: 20%) There exists a nonlinear transformation yi = ϕ(xi) that will allow the Fisher’s linear discriminant and the Logistic regression model to work well on the above data, in the sense that both classification methods will provide 100% classification accuracy on the test data. Write down that data transformation function yi = ϕ(xi).

## 5) (Score: 20%) After applying the nonlinear transformation ϕ(xi) obtained above, draw the transformed data points yi in 1D space. Roughly sketch where the blue squares (training points from C1 class), red diamonds (training points from C2 class), and the black circles (test points) will reside in the transformed space. Also roughly draw the decision boundary of the Logistic regression model applied to this transformed data, and the corresponding decision boundary in the original feature space (Fig. 2). Note that when drawing the transformed data and decision boundaries, just draw their rough arrangement in the new feature space without calculating the exact values.

## 6) (Score: 20%) Which suitable kernel function could be used with Support vector machines to enable 100% classification of the test data points using the available training data? Identify the Support vectors in Fig. 2, which shows data in the original space.

## 7) (Score: 10%) To which class will the test data points a, b, . . . , f be classified using the SVM classifier you used to answer question 6? 

# Topic 4

## 1) (Score: 15%) For the first data point x1 = [0.1, −0.1]T from the training set, what is the output value of the neural network? Show your intermediate calculations.

## 2) (Score: 15%) Assuming the output of the network is y1 = 0.38 given the data point x1. What is the cross entropy error function value E(t1, y1), if the corresponding target is t1 = 1.

## 3) (Score: 40%) Given the data point x1 and the target t1, use backpropagation to calculate the following partial derivatives of the error: ∂E(t1,y1) ∂w(2) 1,1 , ∂E(t1,y1) ∂w(1) 2,4 Show your intermediate calculations.

## 4) (Score: 15%) Briefly explain how you will use ∂E(t1,y1) ∂w(1) 2,4 to update the weight w2,4 of the neural network for the 2nd iteration.

##  5) (Score: 15%) Briefly explain how this network output could be used for a binary classification task, where we must assign a test input xn to either class C0 (tn = 0) or class C1 (tn = 1). Specifically, if the network output for a test input is y = 0.38, which class will you assign to that input and why?

cross entropy error measures probability between 0 and 1, so for binary you check below 0.5 is one class and above 0.5 is another.